In [1]:
import kagglehub
path = kagglehub.dataset_download("umeradnaan/extinction-of-a-species-data")

100%|██████████| 5.29M/5.29M [00:01<00:00, 4.46MB/s]

Extracting files...


In [2]:
import os

# List the contents of the downloaded directory
print(os.listdir(path))


['extinct_species_dataset.csv']


In [3]:
import pandas as pd

df = pd.read_csv(os.path.join(path, 'extinct_species_dataset.csv'))
df.head()

,Species Name,Years Lived (Million Years),Extinction Reason
0,Trilobite,337.75,Human Impact
1,Smilodon,311.57,Natural Disaster
2,Dodo,67.49,Mass Extinction
3,Woolly Mammoth,89.81,Asteroid Impact
4,Woolly Mammoth,395.72,Climate Change


In [4]:
print(df.info())
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 3 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   Species Name                 1000000 non-null  object 
 1   Years Lived (Million Years)  1000000 non-null  float64
 2   Extinction Reason            1000000 non-null  object 
dtypes: float64(1), object(2)
memory usage: 22.9+ MB
None
Species Name                   0
Years Lived (Million Years)    0
Extinction Reason              0
dtype: int64


In [5]:
df_encoded = pd.get_dummies(df, columns=['Species Name', 'Extinction Reason'], drop_first=True)
df_encoded.head()

,Years Lived (Million Years),Species Name_Megalodon,Species Name_Plesiosaur,Species Name_Quagga,Species Name_Sabertooth Tiger,Species Name_Smilodon,Species Name_Steller’s Sea Cow,Species Name_Trilobite,Species Name_Tyrannosaurus Rex,Species Name_Woolly Mammoth,Extinction Reason_Climate Change,Extinction Reason_Habitat Loss,Extinction Reason_Human Impact,Extinction Reason_Mass Extinction,Extinction Reason_Natural Disaster,Extinction Reason_Predation
0,337.75,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False
1,311.57,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False
2,67.49,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
3,89.81,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False
4,395.72,False,False,False,False,False,False,False,False,True,True,False,False,False,False,False


In [6]:
from sklearn.model_selection import train_test_split

# Separate features (X) and target (y)
X = df_encoded.drop(columns=[col for col in df_encoded.columns if 'Extinction Reason' in col])
y = df_encoded[[col for col in df_encoded.columns if 'Extinction Reason' in col]]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (800000, 10)
y_train shape: (800000, 6)
X_test shape: (200000, 10)
y_test shape: (200000, 6)


In [7]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Define the model architecture
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),  # Input layer with the number of features
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(y_train.shape[1], activation='softmax') # Output layer with number of target classes
])

# Compile the model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,054 (39.27 KB)

 Trainable params: 10,054 (39.27 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
history = model.fit(X_train, y_train,
                    epochs=1,  # You can adjust the number of epochs
                    batch_size=32, # You can adjust the batch size
                    validation_split=0.2, # Use 20% of training data for validation
                    verbose=1)

20000/20000 ━━━━━━━━━━━━━━━━━━━━ 31s 2ms/step - accuracy: 0.1659 - loss: 7622.8345 - val_accuracy: 0.2850 - val_loss: 6327.6963


In [10]:
# Save the trained model
model.save('extinction_reason_model.h5')

Now that the model is saved, I will create the Streamlit application. First, I'll install the necessary library.

In [11]:
# Install Streamlit
!pip install streamlit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 60.9 MB/s eta 0:00:00


Next, I'll create a Python file for the Streamlit application. This app will allow users to input species data and get predictions for the extinction reason.

In [12]:
import streamlit as st
import pandas as pd
import tensorflow as tf
import numpy as np

# Load the trained model
model = tf.keras.models.load_model('extinction_reason_model.h5')

# Get the feature names from the training data (excluding target columns)
# This is crucial to ensure consistent input order for prediction
# Assuming X_train has the correct column order
X_train_columns = X_train.columns.tolist()

# Define all possible species names and extinction reasons used during training
# These need to match the columns created by pd.get_dummies
all_species_names = [col.replace('Species Name_', '') for col in df_encoded.columns if 'Species Name_' in col]
all_species_names = ['Trilobite'] + all_species_names # Add one of the original species as a base for drop_first=True

all_extinction_reasons_encoded_cols = [col for col in df_encoded.columns if 'Extinction Reason_' in col]
all_extinction_reasons = [col.replace('Extinction Reason_', '') for col in all_extinction_reasons_encoded_cols]
all_extinction_reasons = ['Human Impact'] + all_extinction_reasons # Add one of the original reasons as a base for drop_first=True

# Streamlit app layout
st.title('Extinction Reason Predictor')
st.write('Enter the details of a species to predict its extinction reason.')

# Input fields
species_name_input = st.selectbox('Species Name', all_species_names)
years_lived_input = st.number_input('Years Lived (Million Years)', min_value=0.0, value=100.0, step=0.1)

# Prepare input for prediction
if st.button('Predict Extinction Reason'):
    # Create an empty DataFrame with all feature columns from training data
    input_df = pd.DataFrame(0, index=[0], columns=X_train_columns)

    # Fill in 'Years Lived (Million Years)'
    input_df['Years Lived (Million Years)'] = years_lived_input

    # Fill in one-hot encoded species name
    if species_name_input != 'Trilobite': # 'Trilobite' would be the reference (dropped_first=True)
        col_name = f'Species Name_{species_name_input}'
        if col_name in input_df.columns:
            input_df[col_name] = True

    # Make prediction
    prediction = model.predict(input_df)
    predicted_class_index = np.argmax(prediction, axis=1)[0]

    # Map the predicted index back to the extinction reason
    # The order of y.columns is important here
    predicted_extinction_reason_col = y.columns[predicted_class_index]
    predicted_reason = predicted_extinction_reason_col.replace('Extinction Reason_', '')

    st.success(f'The predicted extinction reason is: **{predicted_reason}**')

# To run this app, save it as a .py file (e.g., app.py) and run `streamlit run app.py` in your terminal.


2026-08-17 12:38:40.543 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 12:38:41.006 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-08-17 12:38:41.007 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 12:38:41.008 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 12:38:41.009 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 12:38:41.009 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 12:38:41.010 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 12:38:41.011 Thread 'MainThread': mi

In [25]:
import shutil
import os

# The 'path' variable holds the directory where kagglehub downloaded the dataset.
# The CSV file is inside this directory.
source_file_path = os.path.join(path, 'extinct_species_dataset.csv')
destination_file_path = '/content/extinct_species_dataset.csv'

# Copy the dataset to /content/ so the app.py can access it.
shutil.copy(source_file_path, destination_file_path)
print(f"Successfully copied '{source_file_path}' to '{destination_file_path}'")

Successfully copied '/root/.cache/kagglehub/datasets/umeradnaan/extinction-of-a-species-data/versions/1/extinct_species_dataset.csv' to '/content/extinct_species_dataset.csv'


In [28]:
%%writefile app.py
import streamlit as st
import pandas as pd
import tensorflow as tf
import numpy as np
import os

# Load the trained model
model = tf.keras.models.load_model('extinction_reason_model.h5')

# --- Re-load and preprocess data within the Streamlit app context ---
csv_path = '/content/extinct_species_dataset.csv'
df = pd.read_csv(csv_path)
df_encoded = pd.get_dummies(df, columns=['Species Name', 'Extinction Reason'], drop_first=True)

X_columns_for_app = df_encoded.drop(columns=[col for col in df_encoded.columns if 'Extinction Reason' in col]).columns.tolist()
y_columns_for_app = [col for col in df_encoded.columns if 'Extinction Reason' in col]

all_species_names_from_df = df['Species Name'].unique().tolist()
all_extinction_reasons_from_df = [col.replace('Extinction Reason_', '') for col in y_columns_for_app]

# Streamlit app layout
st.set_page_config(layout="centered") # Set layout to wide if preferred
st.title('🌍 Extinction Reason Predictor 🦖')
st.write('Enter the details of a species to predict its extinction reason.')

# Input fields in the sidebar
with st.sidebar:
    st.header('Input Parameters 📊')
    species_name_input = st.selectbox('Species Name', all_species_names_from_df)
    years_lived_input = st.number_input('Years Lived (Million Years)', min_value=0.0, value=100.0, step=0.1)

    # Prepare input for prediction
    if st.button('Predict Extinction Reason'):
        # Create an empty DataFrame with all feature columns in the correct order
        input_df = pd.DataFrame(0, index=[0], columns=X_columns_for_app)

        # Fill in 'Years Lived (Million Years)'
        input_df['Years Lived (Million Years)'] = years_lived_input

        # Fill in one-hot encoded species name
        if species_name_input != all_species_names_from_df[0]: # If the input is not the baseline category
            col_name = f'Species Name_{species_name_input}'
            if col_name in input_df.columns:
                input_df[col_name] = 1 # Use 1 for True in DataFrame

        # Make prediction
        prediction = model.predict(input_df)
        predicted_class_index = np.argmax(prediction, axis=1)[0]

        # Map the predicted index back to the extinction reason
        predicted_extinction_reason_col = y_columns_for_app[predicted_class_index]
        predicted_reason = predicted_extinction_reason_col.replace('Extinction Reason_', '')

        st.success(f'The predicted extinction reason is: **{predicted_reason}**')

st.markdown("""
---
This application predicts the most likely reason for a species' extinction based on its name and how long it lived.
""")


Overwriting app.py


Now that any previous instances are stopped, I will run the Streamlit app again and expose it via a public URL using `ngrok`.

Now, run the Streamlit app using the following command. After executing, click on the public URL that appears to interact with the app.

In [15]:
!pip install --ignore-installed blinker
!pip install streamlit

In [16]:
!pip install streamlit pyngrok

In [17]:
from pyngrok import ngrok
ngrok.set_auth_token("3H2nxZtP4iC5L9tX9K97OPLut9W_4JsZrRVF5aRFQpQCCePy1")

In [29]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://vanity-amperage-commodore.ngrok-free.dev" -> "http://localhost:8501"


In [30]:
# Run the Streamlit app
# This will provide a public URL to access the app
!streamlit run app.py &


2026-08-17 12:48:00.147 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.81.223.96:8501

2026-08-17 12:48:08.388935: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
  Stopping...


In [14]:
!streamlit run app.py &>/dev/null&  # Run in background to avoid blocking the cell
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) ^C
